# Spark DataFrames & Spark SQL - Complete Solution Notebook

In [1]:
# Scenario Statement: Global Streaming Platform OptimizationThe Context: You are a Data Engineer at a global video
# streaming service (like Netflix or YouTube). Every second, your servers generate logs for millions of active video streams.
# These logs contain the "Bandwidth Allocation" (how much data is being sent) for every single user currently watching a video.

# The Problem: Due to a new "Ultra-HD" initiative, the company needs to double the bandwidth allocation for every active stream
#  immediately to improve video quality.

# The Challenge:Massive Data: The number of logs is in the millions, making it impossible for
#   a single computer to process them without crashing or taking hours.

# # Real-Time Speed: The infrastructure costs need to be calculated instantly so server capacity can be adjusted in real-time.
# Mapping the Scenario to the CodeCode StepScenario



from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LabSolution").getOrCreate()

## Load Dataset

In [2]:
df = spark.read.csv("/content/spark_lab_dataset.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()

+--------------+-----------+----------------+------+--------+----------------+--------+
|transaction_id|customer_id|            name|region|  amount|transaction_type|is_fraud|
+--------------+-----------+----------------+------+--------+----------------+--------+
|             1|       1057|Christian Forbes| South|12935.47|        Purchase|       0|
|             2|       1003|   Justin Harris| South|  3709.3|      Withdrawal|       0|
|             3|       1004|   Peter Schmitt| South| 3837.45|        Purchase|       1|
|             4|       1049|  Molly Gonzalez|  East| 46638.1|      Withdrawal|       1|
|             5|       1045| Wanda Mcfarland| North|24212.31|        Purchase|       1|
+--------------+-----------+----------------+------+--------+----------------+--------+
only showing top 5 rows
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- amount: d

## Data Exploration

This cell performs basic data exploration to understand the dataset. It calculates the total number of transactions, lists distinct regions, and counts the occurrences of each transaction type.

In [3]:
print("Total Transactions:", df.count())
df.select("region").distinct().show()
df.groupBy("transaction_type").count().show()

Total Transactions: 500
+------+
|region|
+------+
| South|
|  East|
|  West|
| North|
+------+

+----------------+-----+
|transaction_type|count|
+----------------+-----+
|        Purchase|  162|
|          Refund|  172|
|      Withdrawal|  166|
+----------------+-----+



This cell performs basic data exploration to understand the dataset. It calculates the total number of transactions, lists distinct regions, and counts the occurrences of each transaction type.

## Data Cleaning

This cell cleans the data by dropping any rows that contain null values using `df.dropna()`. It then prints the schema again to confirm that the DataFrame structure remains consistent after cleaning.

In [4]:
df = df.dropna()
df.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- is_fraud: integer (nullable = true)



This cell cleans the data by dropping any rows that contain null values using `df.dropna()`. It then prints the schema again to confirm that the DataFrame structure remains consistent after cleaning.

## Transformations

This cell demonstrates various data transformations. It filters the DataFrame to create subsets for 'high_value' transactions (amount > 10000) and 'fraud_df' (where 'is_fraud' is 1). It also uses `select()` to create a DataFrame with specific columns: customer_id, amount, and region.

In [5]:
high_value = df.filter(df.amount > 10000)
fraud_df = df.filter(df.is_fraud == 1)
selected = df.select("customer_id", "amount", "region")

high_value.show(5)
fraud_df.show(5)
selected.show(5)

+--------------+-----------+----------------+------+--------+----------------+--------+
|transaction_id|customer_id|            name|region|  amount|transaction_type|is_fraud|
+--------------+-----------+----------------+------+--------+----------------+--------+
|             1|       1057|Christian Forbes| South|12935.47|        Purchase|       0|
|             4|       1049|  Molly Gonzalez|  East| 46638.1|      Withdrawal|       1|
|             5|       1045| Wanda Mcfarland| North|24212.31|        Purchase|       1|
|             6|       1045|  Jennifer Casey| South|35929.02|      Withdrawal|       1|
|             7|       1054|   Melissa Klein| North|26482.51|        Purchase|       1|
+--------------+-----------+----------------+------+--------+----------------+--------+
only showing top 5 rows
+--------------+-----------+---------------+------+--------+----------------+--------+
|transaction_id|customer_id|           name|region|  amount|transaction_type|is_fraud|
+---------

This cell demonstrates various data transformations. It filters the DataFrame to create subsets for 'high_value' transactions (amount > 10000) and 'fraud_df' (where 'is_fraud' is 1). It also uses `select()` to create a DataFrame with specific columns: customer_id, amount, and region.

## Aggregations

This cell performs several aggregation operations. It calculates the average transaction amount per region, the total count of fraudulent transactions per region, and the total count for each transaction type.

In [6]:
df.groupBy("region").avg("amount").show()
df.groupBy("region").sum("is_fraud").show()
df.groupBy("transaction_type").count().show()

+------+------------------+
|region|       avg(amount)|
+------+------------------+
| South|24863.821007751936|
|  East|23983.666198347106|
|  West| 25563.42611111112|
| North|23597.939032258077|
+------+------------------+

+------+-------------+
|region|sum(is_fraud)|
+------+-------------+
| South|           57|
|  East|           59|
|  West|           58|
| North|           67|
+------+-------------+

+----------------+-----+
|transaction_type|count|
+----------------+-----+
|        Purchase|  162|
|          Refund|  172|
|      Withdrawal|  166|
+----------------+-----+



This cell performs several aggregation operations. It calculates the average transaction amount per region, the total count of fraudulent transactions per region, and the total count for each transaction type.

## Spark SQL

This section leverages Spark SQL for querying the DataFrame. First, `df.createOrReplaceTempView("transactions")` registers the DataFrame as a temporary SQL table. Then, SQL queries are executed to count transactions per region, find the top 5 transactions by amount, and sum fraudulent transactions per region.

In [7]:
df.createOrReplaceTempView("transactions")

spark.sql("SELECT region, COUNT(*) as total FROM transactions GROUP BY region").show()

spark.sql("SELECT * FROM transactions ORDER BY amount DESC LIMIT 5").show()

spark.sql("SELECT region, SUM(is_fraud) as fraud_count FROM transactions GROUP BY region").show()

+------+-----+
|region|total|
+------+-----+
| South|  129|
|  East|  121|
|  West|  126|
| North|  124|
+------+-----+

+--------------+-----------+-----------------+------+--------+----------------+--------+
|transaction_id|customer_id|             name|region|  amount|transaction_type|is_fraud|
+--------------+-----------+-----------------+------+--------+----------------+--------+
|           427|       1036|      Todd Thomas|  West|49969.04|      Withdrawal|       1|
|           235|       1025|       Mark Smith| South|49825.23|          Refund|       0|
|           311|       1081|          Alan Yu|  West|49818.54|        Purchase|       0|
|           117|       1056|     Thomas Smith| South|49629.78|          Refund|       1|
|           305|       1046|Elizabeth Krueger| North|49602.85|        Purchase|       0|
+--------------+-----------+-----------------+------+--------+----------------+--------+

+------+-----------+
|region|fraud_count|
+------+-----------+
| South|      

This section leverages Spark SQL for querying the DataFrame. First, `df.createOrReplaceTempView("transactions")` registers the DataFrame as a temporary SQL table. Then, SQL queries are executed to count transactions per region, find the top 5 transactions by amount, and sum fraudulent transactions per region.

## Optimization

This cell demonstrates optimization techniques. `df.cache()` caches the DataFrame in memory for faster access in subsequent operations. `df.explain()` shows the physical plan of the DataFrame, which can be useful for understanding and optimizing query execution.

In [8]:
df.cache()
df.explain()

== Physical Plan ==
InMemoryTableScan [transaction_id#17, customer_id#18, name#19, region#20, amount#21, transaction_type#22, is_fraud#23]
   +- InMemoryRelation [transaction_id#17, customer_id#18, name#19, region#20, amount#21, transaction_type#22, is_fraud#23], StorageLevel(disk, memory, deserialized, 1 replicas)
         +- *(1) Filter atleastnnonnulls(7, transaction_id#17, customer_id#18, name#19, region#20, amount#21, transaction_type#22, is_fraud#23)
            +- FileScan csv [transaction_id#17,customer_id#18,name#19,region#20,amount#21,transaction_type#22,is_fraud#23] Batched: false, DataFilters: [atleastnnonnulls(7, transaction_id#17, customer_id#18, name#19, region#20, amount#21, transactio..., Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/spark_lab_dataset.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transaction_id:int,customer_id:int,name:string,region:string,amount:double,transaction_typ...




This cell demonstrates optimization techniques. `df.cache()` caches the DataFrame in memory for faster access in subsequent operations. `df.explain()` shows the physical plan of the DataFrame, which can be useful for understanding and optimizing query execution.

#Scenario Statement: Healthcare Data Surge Optimization
The Context: You are a Data Engineer at a global healthcare analytics company. Every second, hospitals worldwide send patient monitoring logs (heart rate, oxygen levels, blood pressure) into your system.

The Problem: A new global regulation requires doubling the frequency of patient monitoring data collection to ensure higher accuracy in critical care.

The Challenge:

Massive Data: Millions of patient logs per second make it impossible for a single computer to process quickly.

Real‑Time Speed: Doctors need instant dashboards to adjust treatment protocols in ICUs.

### Simulating "Doubling the Frequency" and Real-Time Processing

To simulate doubling the frequency of patient monitoring data collection, we can think of it in terms of generating new data points based on existing ones, or more practically, calculating metrics based on smaller time windows to reflect finer granularity. Here, we'll demonstrate a simple transformation by calculating an average of a metric over a derived 'high-frequency' time interval and adding a 'critical_alert' based on heart rate, simulating an immediate need for analysis.


In [9]:
from pyspark.sql.functions import col, rand, current_timestamp, date_format, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
import random
from datetime import datetime, timedelta

# Define schema for patient monitoring data
patient_schema = StructType([
    StructField("patient_id", IntegerType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("heart_rate", IntegerType(), True),
    StructField("oxygen_level", IntegerType(), True),
    StructField("blood_pressure_systolic", IntegerType(), True),
    StructField("blood_pressure_diastolic", IntegerType(), True)
])

# Generate a large synthetic dataset for patient monitoring logs
num_records = 1000000 # Simulating millions of logs
patient_data = []
start_time = datetime(2023, 1, 1, 0, 0, 0)

for i in range(num_records):
    patient_id = random.randint(1000, 2000)
    # Simulate data points every 30 seconds for a month, then take a random one
    # This is a simplified way to get a timestamp within a range
    log_time = start_time + timedelta(seconds=random.randint(0, 30*24*60*60))
    heart_rate = random.randint(60, 120)
    oxygen_level = random.randint(90, 100)
    blood_pressure_systolic = random.randint(90, 140)
    blood_pressure_diastolic = random.randint(60, 90)
    patient_data.append((patient_id, log_time, heart_rate, oxygen_level, blood_pressure_systolic, blood_pressure_diastolic))

patients_df = spark.createDataFrame(patient_data, patient_schema)

print(f"Generated {patients_df.count()} patient monitoring records.")
patients_df.show(5)
patients_df.printSchema()


Generated 1000000 patient monitoring records.
+----------+-------------------+----------+------------+-----------------------+------------------------+
|patient_id|          timestamp|heart_rate|oxygen_level|blood_pressure_systolic|blood_pressure_diastolic|
+----------+-------------------+----------+------------+-----------------------+------------------------+
|      1976|2023-01-08 03:52:20|       113|          99|                    109|                      77|
|      1120|2023-01-15 22:58:57|       113|          93|                     93|                      61|
|      1406|2023-01-30 06:32:02|        95|          98|                    124|                      83|
|      1424|2023-01-09 16:45:15|       118|          97|                     90|                      87|
|      1831|2023-01-03 14:22:35|        74|          97|                     94|                      60|
+----------+-------------------+----------+------------+-----------------------+------------------------+


In [10]:
from pyspark.sql.functions import avg, when, hour, minute, second

# Add a new column to simulate a 'higher frequency' observation window (e.g., every 15 seconds)
# This is a conceptual representation; in a real-time system, data would be streamed at this frequency.
# For demonstration, we'll derive a '15_sec_interval' ID for grouping.
patients_df_transformed = patients_df.withColumn(
    "15_sec_interval_id",
    (
        hour(col("timestamp")) * 3600 +
        minute(col("timestamp")) * 60 +
        (second(col("timestamp")) / 15).cast("int") * 15
    )
)

# Calculate average heart rate per patient per simulated 15-second interval
# This reflects processing data at a higher frequency
avg_heart_rate_per_interval = patients_df_transformed.groupBy("patient_id", "15_sec_interval_id")\
                                                      .agg(avg("heart_rate").alias("avg_heart_rate_15_sec"))

# Add a 'critical_alert' column based on heart rate (e.g., if avg heart rate is too high/low)
patient_alerts_df = avg_heart_rate_per_interval.withColumn(
    "critical_alert",
    when(col("avg_heart_rate_15_sec") > 100, lit(True))
    .otherwise(lit(False))
)

print("Transformed data with 15-second interval averages and critical alerts:")
patient_alerts_df.show(10)
patient_alerts_df.printSchema()


Transformed data with 15-second interval averages and critical alerts:
+----------+------------------+---------------------+--------------+
|patient_id|15_sec_interval_id|avg_heart_rate_15_sec|critical_alert|
+----------+------------------+---------------------+--------------+
|      1440|             15855|                 61.0|         false|
|      1793|             66615|                110.0|          true|
|      1735|             58560|                100.0|         false|
|      1275|             58065|                 62.0|         false|
|      1270|              3750|                 79.0|         false|
|      1181|             28020|                 90.0|         false|
|      1149|             63240|                 87.0|         false|
|      1117|             66315|                 82.0|         false|
|      1743|             49380|                 97.0|         false|
|      1192|             17130|                 96.0|         false|
+----------+------------------+-

### Optimization and Scalability Considerations

Spark's distributed nature allows processing massive data. For 'real-time speed', techniques like caching, broadcasting, and proper partitioning are crucial. Here, we can cache the transformed DataFrame if it's going to be used repeatedly for dashboard updates or further analysis, ensuring faster access.


In [11]:
# Cache the patient alerts DataFrame for faster subsequent access, mimicking real-time dashboard updates
patient_alerts_df.cache()

print("Number of patient alerts:", patient_alerts_df.count())

# Show only critical alerts
print("Critical alerts:")
patient_alerts_df.filter(col("critical_alert") == True).show(5)

# Explain the plan for a simple query to show Spark's optimization capabilities
print("\nExecution plan for critical alerts:")
patient_alerts_df.filter(col("critical_alert") == True).explain()


Number of patient alerts: 917901
Critical alerts:
+----------+------------------+---------------------+--------------+
|patient_id|15_sec_interval_id|avg_heart_rate_15_sec|critical_alert|
+----------+------------------+---------------------+--------------+
|      1793|             66615|                110.0|          true|
|      1627|              6570|                102.0|          true|
|      1422|              8655|                106.0|          true|
|      1839|             17325|                105.0|          true|
|      1191|             76350|                111.0|          true|
+----------+------------------+---------------------+--------------+
only showing top 5 rows

Execution plan for critical alerts:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Filter critical_alert#499: boolean
   +- InMemoryTableScan [patient_id#454, 15_sec_interval_id#489, avg_heart_rate_15_sec#490, critical_alert#499], [critical_alert#499]
         +- InMemoryRelation [patient_id